# VarQITE Default Calibration

This notebook calibrates **robust VarQITE defaults**.

It focuses on:

- ansatz choice
- imaginary-time step size `dtau`
- total step budget
- seed sensitivity

and ranks settings by accuracy, stability, and runtime.

## How To Use This Notebook

- Edit only the grids in the setup cell.
- Compare at least two molecules before changing global defaults.
- Keep the ansatz list focused on plausible default candidates; this notebook is for default calibration, not arbitrary architecture search.
- Use monotonicity as a stability signal, but prioritize final energy error when differences are large.


In [ ]:
from __future__ import annotations

from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from common import exact_ground_energy_for_problem, summary_stats, timed_call
from qite import run_qite, ANSATZES

MOLECULE_SPECS = [
    {"molecule": "H2", "mapping": "jordan_wigner"},
    {
        "molecule": "LiH",
        "mapping": "jordan_wigner",
        "active_electrons": 2,
        "active_orbitals": 2,
    },
]

DTAU_GRID = [0.2]
STEP_BUDGETS = [75]
SEEDS = [0, 1, 2]

EXACT = {spec["molecule"]: exact_ground_energy_for_problem(**spec) for spec in MOLECULE_SPECS}
TOTAL_RUNS = len(MOLECULE_SPECS) * len(ANSATZES) * len(DTAU_GRID) * len(STEP_BUDGETS) * len(SEEDS)
PROGRESS_EVERY = max(1, TOTAL_RUNS // 12)


In [ ]:
records = []
run_index = 0


for spec in MOLECULE_SPECS:
    shared = dict(spec)
    molecule = shared["molecule"]
    exact_e0 = EXACT[molecule]

    for ansatz in ANSATZES:
        for dtau in DTAU_GRID:
            for steps in STEP_BUDGETS:
                for seed in SEEDS:
                    run_index += 1
                    if run_index == 1 or run_index % PROGRESS_EVERY == 0 or run_index == TOTAL_RUNS:
                        print(
                            f"[{run_index}/{TOTAL_RUNS}] "
                            f"{molecule} | ansatz={ansatz} | dtau={dtau} | "
                            f"steps={steps} | seed={seed}"
                        )
                    result, runtime_s = timed_call(
                        run_qite,
                        suppress_stdout=True,
                        **shared,
                        ansatz_name=ansatz,
                        dtau=dtau,
                        steps=steps,
                        seed=seed,
                        plot=False,
                        show=False,
                        force=False,
                    )
                    energies = np.asarray(result["energies"], dtype=float)
                    monotone_fraction = float(np.mean(np.diff(energies) <= 1e-10)) if energies.size > 1 else 1.0
                    records.append(
                        {
                            "molecule": molecule,
                            "ansatz": ansatz,
                            "dtau": float(dtau),
                            "steps": int(steps),
                            "seed": int(seed),
                            "energy": float(result["energy"]),
                            "abs_error": abs(float(result["energy"]) - exact_e0),
                            "runtime_s": float(runtime_s),
                            "monotone_fraction": monotone_fraction,
                            "num_qubits": int(result["num_qubits"]),
                            "param_count": int(np.asarray(result["final_params"]).size),
                        }
                    )

records_df = pd.DataFrame(records)
print(f"Completed {len(records_df)} VarQITE calibration runs.")
display(records_df.head(10).round(6))


In [ ]:
grouped = defaultdict(list)
for row in records:
    key = (row["molecule"], row["ansatz"], row["dtau"], row["steps"])
    grouped[key].append(row)

summary = []
for key, rows in sorted(grouped.items()):
    errors = [r["abs_error"] for r in rows]
    runtimes = [r["runtime_s"] for r in rows]
    monotone = [r["monotone_fraction"] for r in rows]
    summary.append(
        {
            "molecule": key[0],
            "ansatz": key[1],
            "dtau": key[2],
            "steps": key[3],
            "mean_abs_error": summary_stats(errors)["mean"],
            "std_abs_error": summary_stats(errors)["std"],
            "mean_runtime_s": summary_stats(runtimes)["mean"],
            "mean_monotone_fraction": summary_stats(monotone)["mean"],
            "score": summary_stats(errors)["mean"] + 0.25 * summary_stats(errors)["std"] + 0.02 * (1.0 - summary_stats(monotone)["mean"]) + 0.001 * summary_stats(runtimes)["mean"],
        }
    )

summary_df = pd.DataFrame(summary).sort_values(["molecule", "score", "mean_abs_error", "mean_runtime_s"])
display(summary_df.round(8))


In [ ]:
best_per_molecule = {}
for molecule in sorted({row["molecule"] for row in summary}):
    rows = [row for row in summary if row["molecule"] == molecule]
    rows = sorted(rows, key=lambda row: (row["score"], row["mean_abs_error"], -row["mean_monotone_fraction"]))
    best_per_molecule[molecule] = pd.DataFrame(rows[:10])

for molecule, df in best_per_molecule.items():
    print(f"Top VarQITE settings for {molecule}")
    display(df.round(8))


## How To Interpret The Results

- `best_per_molecule` shows the best local settings for each molecule.
- `recommended_defaults` intentionally restricts the aggregate recommendation to `UCCSD`, because that is the chemistry-first default path in this repo.
- If a larger `dtau` wins only by a negligible error margin, prefer the smaller and more conservative value as the package default.


In [ ]:
recommended_defaults = []
candidate_rows = [row for row in summary if row["ansatz"] == "UCCSD"]
if candidate_rows:
    by_setting = defaultdict(list)
    for row in candidate_rows:
        by_setting[(row["dtau"], row["steps"])].append(row)

    aggregate = []
    for key, rows in sorted(by_setting.items()):
        aggregate.append(
            {
                "dtau": key[0],
                "steps": key[1],
                "mean_abs_error": float(np.mean([row["mean_abs_error"] for row in rows])),
                "mean_std_abs_error": float(np.mean([row["std_abs_error"] for row in rows])),
                "mean_monotone_fraction": float(np.mean([row["mean_monotone_fraction"] for row in rows])),
                "mean_runtime_s": float(np.mean([row["mean_runtime_s"] for row in rows])),
            }
        )
    aggregate = sorted(
        aggregate,
        key=lambda row: (row["mean_abs_error"] + 0.25 * row["mean_std_abs_error"] + 0.02 * (1.0 - row["mean_monotone_fraction"]) + 0.001 * row["mean_runtime_s"], row["mean_abs_error"]),
    )
    recommended_defaults = aggregate[:10]

recommended_defaults_df = pd.DataFrame(recommended_defaults)
print("Recommended VarQITE default candidates (UCCSD-only aggregate)")
display(recommended_defaults_df.round(8))


In [ ]:
for molecule in sorted(best_per_molecule):
    rows = best_per_molecule[molecule][:5]
    labels = [f"{row['ansatz']}\ndtau={row['dtau']}\nN={row['steps']}" for row in rows]
    values = [row["mean_abs_error"] for row in rows]

    plt.figure(figsize=(9, 4))
    plt.bar(labels, values, alpha=0.85)
    plt.ylabel("Mean absolute error (Ha)")
    plt.title(f"Top VarQITE default candidates for {molecule}")
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()